## 1. Вопросы

### i. Что такое leave-one-out? Преимущества и недостатки

Это частный случай кросс-валидации, где число фолдов равно общему количеству объектов в датасете N. Всего N итераций в цикле, в каждой из которой 1 строка - валидация, а остальные N-1 - обучающая часть

Преимущества:

* Максимально эффективное использование данных - почти вся выборка идет на обучение

* Детерминированность результата (нет случайности в разбиении)

Недостатки:

* Высокая вычислительная стоимость при большом N (обучение модели N раз)

* Высокая дисперсия оценки - каждый раз модель тестируется на одном объекте, который может оказаться выбросом (шумная оценка)

### ii. Как работают Grid Search, Randomized Grid Search, и Байесовская оптимизация?

1.  **Grid Search**
    Для каждого гиперпараметра фиксируется несколько значений. Далее перебираются все комбинации значений различных гиперпараметров, на каждой из этих комбинаций модель обучается и тестируется.

2.  **Randomized Grid Search**
    Значения - распределения вероятностей для каждого гиперпараметра. Случайно выбираются значения в заданном диапазоне и выполняется подбор выбранное количество раз.

3.  **Байесовская оптимизация**
     это метод глобальной оптимизации «черных ящиков» - функций, которые дорого вычислять. Она использует результаты прошлых попыток.

    *   **f(x)** - целевая функция
    *   **X** - пространство всех возможных параметров
    *   **Dn = {(x₁, y₁), (x₂, y₂), ..., (xₙ, yₙ)}** - это набор уже проверенных точек, где **yₙ = f(xₙ)** - реальная точность, полученная после обучения модели
    *   **n** - счетчик итераций

    **Шаг 0: Инициализация**

    Мы ничего не знаем о функции f. Поэтому берем **n₀** случайных комбинаций гиперпараметров:

    **x₁, x₂, ..., xₙ₀** (выбираем случайно из пространства X)

    Для каждой из них обучаем реальную модель и получаем точность:

    **y₁ = f(x₁), y₂ = f(x₂), ..., yₙ₀ = f(xₙ₀)**

    Составляем начальный датасет:

    **D = {(x₁, y₁), (x₂, y₂), ..., (xₙ₀, yₙ₀)}**

    **Шаг 1: Строим сюррогатную модель**

    На основе имеющихся данных D мы строим модель-заменитель. Для каждой новой точки x эта модель предсказывает не одно значение, а распределение:

    **f(x) ∼ GP(m(x), k(x, x'))**

    где:

    $m(\mathbf{x})$ - функция среднего (обычно 0 после центрирования)

    $k(\mathbf{x}, \mathbf{x}')$ - ковариационная функция (ядро)

    Самое популярное - **RBF-ядро** (Radial Basis Function), квадратичное экспоненциальное:

    $$k_{RBF}(x, x') = \sigma_f^2 \exp\left(-\frac{||x - x'||^2}{2l^2}\right)$$

    Параметры ядра (гиперпараметры GP):
    *   **$l$ (length scale)** - определяет, как далеко распространяется влияние точки
    *   **$\sigma_f^2$** - сигнал дисперсии (априорная вариативность функции)

    **Шаг 2: Выбираем следующую точку (acquisition function)**

    **критерий сбора данных (acquisition function)** α(x). Он комбинирует μ(x) и σ(x) в одно число.

    Самый популярный критерий - **Expected Improvement (EI)**. Его формула:

    $$EI(x) = \mathbb{E}[\max(0, f(x) - f_{best})]$$

    Если расписать для нормального распределения:

    $$EI(x) = (f_{best} - \mu(x)) \cdot \Phi(z) + \sigma(x) \cdot \phi(z)$$

    где:

    $$z = \frac{f_{best} - \mu(x)}{\sigma(x)}$$

    *   **Φ** - функция нормального распределения (вероятность)
    *   **φ** - плотность нормального распределения
    *   **f_best** - лучший результат на данный момент (максимальное значение среди наблюдаемых y)

    Следующая точка выбирается там, где EI максимален:

    $$x_{new} = \arg\max_{x \in X} EI(x)$$

    **Шаг 3: Проверяем реальную функцию**

    В выбранной точке **x_new** мы обучаем реальную ML-модель и получаем реальную точность:

    $$y_{new} = f(x_{new})$$

    **Шаг 4: Обновляем данные**

    Добавляем новую пару в историю наблюдений:

    $$D = D \cup \{(x_{new}, y_{new})\}$$

    Увеличиваем счетчик итераций:

    $$n = n + 1$$

    Возвращаемся к **Шагу 1**. Цикл повторяется, пока не закончится бюджет итераций.

### iii. Объясните классификацию методов выбора признаков. Объясните, как работают Pearson и Chi2. Объясните, как работает Lasso. Объясните, что такое значение перестановки. Познакомьтесь с SHAP.

#### Методы отбора признаков

Методы отбора признаков обычно делят на **Unsupervised** и **Supervised**.

#### 1. Unsupervised (без учителя)

Используются без учета целевой переменной. Идея - удалить «плохие» или избыточные признаки на основе их свойств. Эти методы не учитывают, насколько признак полезен для предсказания.

- **Удаление признаков с пропусками** (drop incomplete features)
- **Удаление признаков с высокой мультиколлинеарностью** (сильно коррелирующих друг с другом)
- **Удаление признаков с почти нулевой дисперсией** (near-zero variance)

#### 2. Supervised (с учителем)

Используют информацию о целевой переменной. Делятся на три большие группы:

##### 2.1 Wrappers (обёрточные методы)

**Идея:** обучаем модель много раз и выбираем лучший набор признаков по качеству модели.

- Forward selection (пошаговое добавление)
- Backward elimination (пошаговое удаление)
- Recursive Feature Elimination (RFE)

##### 2.2 Filters (фильтры)

**Идея:** оцениваем каждый признак отдельно с помощью статистики, без обучения модели.

- Pearson’s r
- Kendall Tau
- Spearman’s rho
- Chi²
- Mutual Information
- F-score
- Point-biserial

##### 2.3 Embedded (встроенные методы)

Отбор происходит во время обучения модели.

- LASSO (L1-регуляризация)
- Autoencoder с bottleneck-слоем

#### Pearson

**Ковариация**

Связь:
$$x > \bar{x} \to y > \bar{y}$$
$$x < \bar{x} \to y < \bar{y}$$

$$
\text{cov}(X,Y) = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{n-1}
$$

Деление на $$(n-1)$$ используется, чтобы получить несмещённую оценку выборочной ковариации. 

> Важно: Ковариация зависит от масштаба данных.

**Нормализация**

Стандартное отклонение:

$$
\sigma_x = \sqrt{\frac{1}{n-1}\sum (x_i - \bar{x})^2}
$$

**Коэффициент корреляции Пирсона:**

$$
r = \frac{\text{cov}(X,Y)}{\sigma_x \sigma_y}
$$

*   Не зависит от масштаба данных.
*   Принимает значения в диапазоне [-1, 1].

**Векторная интерпретация**

После центрирования (вычитания среднего) данные можно представить как векторы в n-мерном пространстве:

$$
\vec{X} = (x_1 - \bar{x}, \dots, x_n - \bar{x})
$$
$$
\vec{Y} = (y_1 - \bar{y}, \dots, y_n - \bar{y})
$$

Тогда:

*   Ковариация пропорциональна скалярному произведению:
    $$
    \text{cov}(X,Y) = \frac{1}{n-1} (\vec{X} \cdot \vec{Y})
    $$

*   Коэффициент корреляции - это косинус угла между векторами:
    $$
    r = \cos(\theta)
    $$

**Интерпретация через угол**

*   $\theta = 0\circ \to r = 1$: идеальная положительная связь (векторы сонаправлены).
*   $\theta = 180^\circ \to r = -1$: идеальная отрицательная связь (векторы противоположно направлены).
*   $\theta = 90^\circ \to r = 0$: нет линейной связи (векторы ортогональны).

$$
|r| \le 1
$$

**Ключевой вывод: Корреляция - это косинус угла между центрированными векторами.**

#### Chi2

Проверяем, зависит ли категориальный признак \(X\) от категориального признака \(Y\).

**Статистика критерия**

$$
\chi^2 = \sum \frac{(O - E)^2}{E}
$$

- O - наблюдаемое значение
- E - ожидаемое значение  

Статистика $\chi^2$ измеряет, насколько фактические частоты отличаются от ожидаемых при независимости признаков.

**Ожидаемые частоты**

При независимости:

$$
P(X,Y) = P(X)\cdot P(Y)
$$

где \(N\) - число объектов.

$$
P(X) = \frac{\text{сумма строки}}{N}
$$

$$
P(Y) = \frac{\text{сумма столбца}}{N}
$$

Тогда ожидаемое значение:

$$
E_{ij} = N \cdot P(X_i)\cdot P(Y_j)
$$

или

$$
E_{ij} =
\frac{(\text{сумма строки}_i)\cdot(\text{сумма столбца}_j)}{N}
$$

#### Lasso

Lasso выполняет отбор признаков.

Модель:

$$
y = X\beta + \varepsilon
$$

**Обычный метод наименьших квадратов**

$$
\min_{\beta} \sum (y_i - x_i\beta)^2
$$

или

$$
\min_{\beta} \|y - X\beta\|^2
$$

**Lasso-регуляризация**

$$
\min_{\beta} \left(\|y - X\beta\|^2 + \lambda \|\beta\|_1 \right)
$$

где

$$
\|\beta\|_1 = \sum |\beta_j|
$$

**Ограничение L1**

$$
|\beta_1| + |\beta_2| \le t
$$

Множество допустимых значений для L1-нормы образует **ромб**.

- t - число, которое зависит от $\lambda$ и масштаба данных. Радиус L1-области.

**Геометрическая интерпретация**

- минимум квадратичной ошибки - **эллипс**
- область L1 - **ромб**

Эллипс часто касается угла ромба, где один из коэффициентов равен нулю.

Поэтому Lasso зануляет некоторые коэффициенты и выполняет отбор признаков.

#### Значение перестановки (permutation importance)

Это мера того, насколько ухудшается качество модели, если случайно перемешать значения одного признака.

- Если качество сильно падает -> признак важный.
- Если почти не меняется -> признак неважный.

Если признак действительно влияет на предсказание:

- модель использует его структуру -> если мы её разрушим (перемешаем значения) -> модель начинает ошибаться


#### SHAP (Shapley Additive Explanations)

Метод объяснения предсказаний модели, основанный на теории игр (значения Шепли).

**Идея**: Разложить предсказание модели на вклад каждого признака.

**Модель**
$$
f(x) = \varphi_0 + \sum_{j=1}^{p} \varphi_j
$$

- $\varphi_0$ — среднее предсказание модели (baseline)
- $\varphi_j$ — вклад признака j: насколько признак изменил предсказание относительно baseline

**Подсчёт вкладов**

Для каждого признака:

1. Берём все возможные подмножества других признаков
2. Смотрим, как меняется предсказание, если добавить признак к этому подмножеству
3. Усредняем вклад по всем возможным порядкам добавления

**Формула значения Шепли**

$$
\varphi_j(v) =
\sum_{S \subseteq N \setminus \{j\}}
\frac{|S|!(|N| - |S| - 1)!}{|N|!}
\left( v(S \cup \{j\}) - v(S) \right)
$$

**Обозначения**

- N — множество всех признаков  
- |N| — мощность множества
- S — подмножество признаков без признака j 
- v(S) — предсказание модели для подмножества признаков  
- $v(S \cup \{j\})$ — предсказание с добавленным признаком \(j\)

## 2. Обработка данных

In [7]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from itertools import product
from sklearn.base import clone
import optuna
import shap
import time

In [8]:
train_df = pd.read_json("validation-optimization/data/train.json")
test_df = pd.read_json("validation-optimization/data/test.json")
train_df.head(3)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium


In [9]:
def clean_features(features_list):
    clean_features = []
    for feature in features_list:
        cleaned = feature.replace(" ", "")
        clean_features.append(cleaned)
    return clean_features

In [10]:
train_df["clean_features"] = train_df['features'].apply(clean_features)
print(train_df['clean_features'])

4         [DiningRoom, Pre-War, LaundryinBuilding, Dishw...
6         [Doorman, Elevator, LaundryinBuilding, Dishwas...
9         [Doorman, Elevator, LaundryinBuilding, Laundry...
10                                                       []
15        [Doorman, Elevator, FitnessCenter, LaundryinBu...
                                ...                        
124000               [Elevator, Dishwasher, HardwoodFloors]
124002    [CommonOutdoorSpace, CatsAllowed, DogsAllowed,...
124004    [DiningRoom, Elevator, Pre-War, LaundryinBuild...
124008    [Pre-War, LaundryinUnit, Dishwasher, NoFee, Ou...
124009    [DiningRoom, Elevator, LaundryinBuilding, Dish...
Name: clean_features, Length: 49352, dtype: object


In [11]:
def collected_features(df):
    all_features = []
    for _, row in df.iterrows():
        features_list = row['clean_features']
        all_features.extend(features_list)
    return all_features

all_features = collected_features(train_df)
print(len(all_features))

267906


In [12]:
print(f"Количество уникальных значений: {len(set(all_features))}")

Количество уникальных значений: 1548


In [13]:
def get_top_features(features, n):
    count_features = Counter(features)
    top_features = count_features.most_common(n)
    top_features_name = [name for name, count in top_features]
    return top_features_name

top_features = get_top_features(all_features, 20)
top_features

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [14]:
def has_feature(names, df):
    for feature_name in names:
        new_column_name = f'has_{feature_name}'
        df[new_column_name] = df['clean_features'].apply(
            lambda x: 1 if feature_name in x else 0
        )
    return df

train_df = has_feature(top_features, train_df)
train_df.head(3)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,...,has_LaundryinUnit,has_RoofDeck,has_OutdoorSpace,has_DiningRoom,has_HighSpeedInternet,has_Balcony,has_SwimmingPool,has_LaundryInBuilding,has_NewConstruction,has_Terrace
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,...,1,0,0,0,0,0,0,0,0,0


In [15]:
def create_final_list(df):
    feature_list = ['bathrooms', 'bedrooms', 'created']
    columns = df.columns
    for col in columns:
        if 'has_' in col:
            feature_list.append(col)
    return feature_list

feature_list = create_final_list(train_df)
X = train_df[feature_list].copy()
y = train_df['price']
display(X.head(3))


,bathrooms,bedrooms,created,has_Elevator,has_CatsAllowed,has_HardwoodFloors,has_DogsAllowed,has_Doorman,has_Dishwasher,has_NoFee,...,has_LaundryinUnit,has_RoofDeck,has_OutdoorSpace,has_DiningRoom,has_HighSpeedInternet,has_Balcony,has_SwimmingPool,has_LaundryInBuilding,has_NewConstruction,has_Terrace
4,1.0,1,2016-06-16 05:55:27,0,1,1,1,0,1,0,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,2016-06-01 05:44:33,1,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,2016-06-14 15:19:59,1,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,0


## 3.1 Разделите данные на 2 части случайным образом с параметром test_size

In [16]:
def my_train_test_split(X, y=None, test_size=0.2, random_state=None):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)

    indx = np.arange(n_samples)
    np.random.shuffle(indx)

    if 0 < test_size < 1:
        test_end = int(n_samples * test_size)
    else:
        raise ValueError(f"test_size must be between 0 and 1, got {test_size}")
    
    test_indx = indx[:test_end]
    train_indx = indx[test_end:]

    def split_data(data, train_idx, test_idx):
        if isinstance(data, pd.DataFrame):
            return data.iloc[train_idx], data.iloc[test_idx]
        else: 
            return data[train_idx], data[test_idx]
        
    if y is not None:
        if len(y) != len(X):
            raise ValueError(f"X and y must have same length: {n_samples} vs {len(y)}")
        X_train, X_test = split_data(X, train_indx, test_indx)
        y_train, y_test = split_data(y, train_indx, test_indx)
        return X_train, X_test, y_train, y_test
    else:
        return split_data(X, train_indx, test_indx)


In [17]:
X_temp, X_test, y_temp, y_test = my_train_test_split(X, y, test_size=0.2, random_state=42)

## 3.2 Случайным образом разделите данные на 3 части

In [18]:
def train_val_test_split(X, y=None, test_size=0.2, val_size = 0.25, random_state=None):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    if random_state is not None:
        np.random.seed(random_state)

    n_samples = len(X)

    indx = np.arange(n_samples)
    np.random.shuffle(indx)

    if 0 < (test_size + val_size) < 1:
        test_end = int(n_samples * test_size)
        val_end = int(n_samples * val_size) + test_end
    else:
        raise ValueError(f"test_size and val_size must be between 0 and 1 and their sum is less than 1, got {test_size},{val_size}")
    
    test_indx = indx[:test_end]
    val_indx = indx[test_end:val_end]
    train_indx = indx[val_end:]
    
    def split_data(data, train_idx, val_idx, test_idx):
        if isinstance(data, pd.DataFrame) or isinstance(data, pd.Series):
            return data.iloc[train_idx], data.iloc[val_idx],data.iloc[test_idx]
        else: 
            return data[train_idx], data[val_idx],data[test_idx]
        
    if y is not None:
        if len(y) != len(X):
            raise ValueError(f"X and y must have same length: {n_samples} vs {len(y)}")
        X_train, X_val, X_test = split_data(X, train_indx, val_indx, test_indx)
        y_train, y_val, y_test = split_data(y, train_indx, val_indx, test_indx)
        return X_train, X_val, X_test, y_train, y_val, y_test
    else:
        return split_data(X, train_indx, val_indx, test_indx)


In [19]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, random_state=42)

## 3.3 Разделите данные на 2 части с помощью параметра date_split

In [20]:
def train_test_date_split(df, sort_by, target_col, date_split = 0.2): 
    df = df.copy()
    if isinstance(sort_by, str):   
        df[sort_by] = pd.to_datetime(df[sort_by])
        df.sort_values(sort_by, inplace=True)
    else:
        raise ValueError("sort_by must be string")
    
    if not 0 < date_split < 1:
        raise ValueError(f"date_split must be between 0 and 1, got {date_split}")

    split_indx = int(len(df) * date_split)

    test = df.iloc[:split_indx]
    train = df.iloc[split_indx:]

    if isinstance(target_col, str):
        X_train, y_train = train.drop(target_col, axis=1), train[target_col]
        X_test, y_test = test.drop(target_col, axis=1), test[target_col]
    else:
        raise ValueError("target_col must be string")

    return X_train, X_test, y_train, y_test

In [21]:
X_train, X_test, y_train, y_test = train_test_date_split(train_df, 'created', 'price')

print(X_train, X_test, y_train, y_test)

        bathrooms  bedrooms                       building_id  \
90303         2.0         3  c65d06a3598abe4da4509b8243bde68e   
105451        1.0         3  b8e75fc949a6cd8225b455648a951712   
111604        2.0         2  f5e2cdb7a92059ee5e3dbd97b03f03ce   
120649        2.0         2  8b011d67e60a8c802192a1d7816dea85   
95101         1.0         3  a3eb10802f4ade9293e05f1ac6e79a9a   
...           ...       ...                               ...   
335           1.0         2  b2cc0f022ed6e0909559b6f4ea2c93df   
26349         1.0         1  8c86e6c7f2fca184602f484168e2c9a6   
622           1.0         1  a61b2882bb7a66c691471523811b19b8   
34914         1.0         3  2d4de807c29963b5aac44d4300e35dc9   
28466         1.0         3  2d4de807c29963b5aac44d4300e35dc9   

                   created                                        description  \
90303  2016-04-19 03:57:40                                                      
105451 2016-04-19 03:57:46          BRAND NEW GUT RENOVAT

## 3.4 Разделите данные на 3 части с параметрами validation_date и test_date

In [22]:
def train_val_test_date_split(X, y, sort_by, val_date = 0.2, test_date=0.2): 
    if not 0 < val_date + test_date < 1:
        raise ValueError(f"val_date and test_date must be between 0 and 1, got {val_date}, {test_date}")

    if not isinstance(sort_by, str):  
        raise ValueError("sort_by must be string")

    if y.name is None:
        raise ValueError("y must have a name attribute")

    df = X.copy()
    df[sort_by] = pd.to_datetime(df[sort_by])
    df[y.name] = y.values
    df = df.sort_values(sort_by).reset_index(drop=True)
    
    train_indx = int(len(df) * (1 - test_date - val_date))
    val_indx = train_indx +int(len(df) * val_date)

    X_sort = df.drop(y.name, axis=1)
    y_sort = df[y.name]
    
    X_train, y_train = X_sort.iloc[:train_indx], y_sort[:train_indx]
    X_val, y_val = X_sort.iloc[train_indx:val_indx], y_sort[train_indx:val_indx]
    X_test, y_test = X_sort.iloc[val_indx:], y_sort[val_indx:]

    return X_train, X_val, X_test, y_train, y_val, y_test

In [23]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_date_split(X, y, 'created')

print(X_train, X_val, X_test, y_train, y_val, y_test)

       bathrooms  bedrooms             created  has_Elevator  has_CatsAllowed  \
0            1.0         1 2016-04-01 22:12:41             1                0   
1            1.0         0 2016-04-01 22:56:00             0                1   
2            2.0         3 2016-04-01 22:57:15             1                1   
3            1.0         1 2016-04-01 23:26:07             1                1   
4            1.0         1 2016-04-02 00:48:13             1                1   
...          ...       ...                 ...           ...              ...   
29606        1.0         2 2016-05-24 20:23:36             0                1   
29607        1.0         1 2016-05-24 20:53:21             0                1   
29608        1.0         2 2016-05-24 21:52:30             1                1   
29609        1.0         0 2016-05-24 22:30:51             1                1   
29610        2.0         4 2016-05-25 01:10:42             0                0   

       has_HardwoodFloors  

## 3.5 Сделайте процедуру разделения детерминированной. Что это значит?

Deterministic (детерминированный) означает, что при каждом запуске кода с одними и теми же входными данными будет получаться строго одинаковый результат

## 4.1 K-Fold CV

In [24]:
def kfold_split(X, k = 5, shuffle = True, random_state = None):
    n_samples = len(X)
    indx = np.arange(n_samples)

    if shuffle:
        if random_state is not None:
            np.random.seed(random_state)
        np.random.shuffle(indx)

    fold_size = n_samples // k
    folds = []

    for i in range(k):
        start = i * fold_size
        end = (i + 1) * fold_size if i < k - 1 else n_samples

        test_idx = indx[start:end]
        train_idx = np.concatenate([indx[:start], indx[end:]])
        
        folds.append((train_idx, test_idx))
    
    return folds

In [25]:
folds = kfold_split(X, k=3)

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}:")
    print(f"Train indices: {train_idx}")
    print(f"Test indices: {test_idx}")
    print(f"Train data: {X.iloc[train_idx]}")
    print(f"Test data: {X.iloc[test_idx]}")

Fold 1:
Train indices: [31320 15385 35119 ... 45654 33782 16000]
Test indices: [42676 28057 24528 ... 39471 28747 37858]
Train data:         bathrooms  bedrooms              created  has_Elevator  \
78708         2.0         2  2016-05-18 02:31:11             1   
38672         1.0         2  2016-06-01 17:29:32             1   
88337         2.0         2  2016-04-20 02:49:22             1   
18001         1.0         1  2016-06-12 01:22:35             0   
20289         1.0         3  2016-06-05 22:28:37             0   
...           ...       ...                  ...           ...   
53560         1.0         2  2016-05-24 03:42:30             1   
25148         2.0         3  2016-06-25 02:27:54             1   
114840        1.0         1  2016-04-29 13:49:38             0   
84807         1.0         2  2016-04-26 02:31:11             0   
40208         1.0         1  2016-06-05 04:16:09             1   

        has_CatsAllowed  has_HardwoodFloors  has_DogsAllowed  has_Doorman 

## 4.2 Grouped K-Fold CV

In [26]:
def grouped_split(group_field, k = 5, shuffle = True, random_state = None):
    group_field = np.array(group_field)
    unique_groups = np.unique(group_field)
    n_groups = len(unique_groups)

    if k > n_groups:
        raise ValueError(f"k={k} > n_groups={n_groups}")
    
    if shuffle:
        if random_state is not None:
            np.random.seed(random_state)
        np.random.shuffle(unique_groups)

    groups_in_folds = [[] for _ in range(k)]
    
    for i, group in enumerate(unique_groups):
        fold_idx = i % k
        groups_in_folds[fold_idx].append(group)

    folds = []
    for i in range(k):
        test_groups = set(groups_in_folds[i])
        train_groups = set()
        for j in range(k):
            if j != i:
                train_groups.update(groups_in_folds[j])

        test_idx = [idx for idx, g in enumerate(group_field) if g in test_groups]
        train_idx = [idx for idx, g in enumerate(group_field) if g in train_groups]

        folds.append((train_idx, test_idx))

    return folds

In [27]:
folds = grouped_split(X['bedrooms'], k=3)

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"Fold {i+1}:")
    print(f"Train groups: {np.unique(X['bedrooms'].iloc[train_idx])}")
    print(f"Test groups: {np.unique(X['bedrooms'].iloc[test_idx])}")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")

Fold 1:
Train groups: [1 2 3 4 5 6]
Test groups: [0 7 8]
Train size: 39873, Test size: 9479
Fold 2:
Train groups: [0 3 4 5 7 8]
Test groups: [1 2 6]
Train size: 18931, Test size: 30421
Fold 3:
Train groups: [0 1 2 6 7 8]
Test groups: [3 4 5]
Train size: 39900, Test size: 9452


## 4.3 Stratified K-fold CV

In [28]:
def stratified_split(stratify_field, k, shuffle = True, random_state=None):
    y = np.array(stratify_field)
    unique_classes = np.unique(y)

    class_indx = {c: np.where(y == c)[0] for c in unique_classes}

    if shuffle:
        if random_state is not None:
            np.random.seed(random_state)
        for c in unique_classes:
            np.random.shuffle(class_indx[c])

    fold_indx = [[] for _ in range(k)]

    for c in unique_classes:
        indices = class_indx[c]
        n_class = len(indices)

        fold_sizes = np.full(k, n_class // k, dtype=int)
        fold_sizes[:n_class%k] += 1

        current = 0
        for fold_idx, fold_size in enumerate(fold_sizes):
            fold_indx[fold_idx].extend(indices[current:current + fold_size])
            current += fold_size

    folds = []
    for i in range(k):
        test_idx = fold_indx[i]
        train_idx = np.concatenate([fold_indx[j] for j in range(k) if j != i])

        folds.append((train_idx.tolist(), [int(x) for x in test_idx]))

    return folds

In [29]:
y_temp = np.array([0, 1, 0, 2, 0, 1, 2, 0])
train_test_indx = stratified_split(y_temp, k=2)
print(train_test_indx)

[([2, 4, 5, 6], [0, 7, 1, 3]), ([0, 7, 1, 3], [2, 4, 5, 6])]


## 4.4 Time series split

In [30]:
def time_series_split(X, k, date_field):
    n_samples = len(X)
    dates = np.array(date_field)

    sorted_indx = np.argsort(dates)

    test_size = n_samples //(k+1)

    folds = []
    for i in range(1, k+1):
        train_end = i * test_size
        test_start = train_end
        test_end = min(test_start + test_size, n_samples)

        if test_start >= n_samples:
            break

        train_idx = sorted_indx[:train_end]
        test_idx = sorted_indx[test_start:test_end]

        folds.append((train_idx.tolist(), test_idx.tolist()))

    return folds

## 5. Сравнение CV

In [31]:
def compare_implement(X, y, groups, dates, k=5):

    results = {}

    print("K-Fold CV")
    our_kfolds = kfold_split(X, k, shuffle=True, random_state=42)
    sklearn_kfolds = KFold(n_splits=k, shuffle=True, random_state=42)
    sklearn_folds = list(sklearn_kfolds.split(X))

    print(f"Количество фолдов: {k}")
    print(f"Наши фолды: {len(our_kfolds)}")
    print(f"Sklearn фолды: {len(sklearn_folds)}")

    our_train, our_test = our_kfolds[0]
    sk_train, sk_test = sklearn_folds[0]

    print(f"Фолд 1 - размер train: наш={len(our_train)}, sklearn={len(sk_train)}")
    print(f"Фолд 1 - размер test: наш={len(our_test)}, sklearn={len(sk_test)}")
    
    print(f"Совпадение train индексов: {set(our_train) == set(sk_train)}")
    print(f"Совпадение test индексов: {set(our_test) == set(sk_test)}")

    results['kfold'] = {'our': our_kfolds, 'sklearn': sklearn_folds, 
                        'match': set(our_train)==set(sk_train) and set(our_test)==set(sk_test)}
    
    print("StratifiedKFold")

    our_strat = stratified_split(y, k, shuffle=True, random_state=42)
    sk_strat = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    sk_strat_folds = list(sk_strat.split(X, y))

    our_train, our_test = our_strat[0]
    sk_train, sk_test = sk_strat_folds[0]

    print(f"Фолд 1 - размер train: наш={len(our_train)}, sklearn={len(sk_train)}")
    print(f"Фолд 1 - размер test: наш={len(our_test)}, sklearn={len(sk_test)}")

    our_y_train, our_y_test = y[our_train], y[our_test]
    sk_y_train, sk_y_test = y[sk_train], y[sk_test]

    print(f"\nРаспределение классов в train (наша): {Counter(our_y_train)}")
    print(f"Распределение классов в train (sklearn): {Counter(sk_y_train)}")
    print(f"Распределение классов в test (наша): {Counter(our_y_test)}")
    print(f"Распределение классов в test (sklearn): {Counter(sk_y_test)}")
    
    results['stratified'] = {'our': our_strat, 'sklearn': sk_strat_folds,
                             'match': set(our_train) == set(sk_train) and set(our_test) == set(sk_test)}

    print("GroupKFold")
    
    our_group = grouped_split(groups, k)
    sk_group = GroupKFold(n_splits=k)
    sk_group_folds = list(sk_group.split(X, y, groups))

    our_train, our_test = our_group[0]
    sk_train, sk_test = sk_group_folds[0]

    print(f"\nФолд 1 - размер train: наш={len(our_train)}, sklearn={len(sk_train)}")
    print(f"Фолд 1 - размер test: наш={len(our_test)}, sklearn={len(sk_test)}")

    our_test_groups= set(groups.iloc[our_test])
    sk_test_groups = set(groups.iloc[sk_test])

    print(f"Уникальные группы в test (наша): {our_test_groups}")
    print(f"Уникальные группы в test (sklearn): {sk_test_groups}")

    results['group'] = {'our': our_group, 'sklearn': sk_group_folds,
                        'match': our_test_groups == sk_test_groups}
    
    print("TimeSeriesSplit")

    our_ts = time_series_split(X, k, dates)
    sk_ts = TimeSeriesSplit(n_splits=k)
    sk_ts_folds = list(sk_ts.split(X))

    for i in range(min(len(our_ts), len(sk_ts_folds))):
        our_train, our_test = our_ts[i]
        sk_train, sk_test = sk_ts_folds[i]

        print(f"Фолд: {i+1}")
        print(f"Train размер: наш={len(our_train)}, sklearn={len(sk_train)}")
        print(f"Test размер: наш={len(our_test)}, sklearn={len(sk_test)}")

        our_train_dates = dates[our_train]
        our_test_dates = dates[our_test]

        print(f"Train dates: {our_train_dates[0]} ... {our_train_dates[-1]}")
        print(f"Test dates: {our_test_dates[0]} ... {our_test_dates[-1]}")

        print(f"Train раньше Test: {max(our_train_dates) < min(our_test_dates)}")

    results['timeseries'] = {'our': our_ts, 'sklearn': sk_ts_folds,
                                'match': set(our_train)==set(sk_train) and set(our_test)==set(sk_test)}
    
    return results

In [32]:
groups = train_df['bedrooms'].copy()
X = train_df[feature_list].copy()
y = train_df['price'].copy()
dates = train_df['created'].values
y_disc = pd.cut(pd.Series(y), bins=3, labels=False).astype(int).values
res = compare_implement(X, y_disc, groups, dates, k=3)

K-Fold CV
Количество фолдов: 3
Наши фолды: 3
Sklearn фолды: 3
Фолд 1 - размер train: наш=32902, sklearn=32901
Фолд 1 - размер test: наш=16450, sklearn=16451
Совпадение train индексов: False
Совпадение test индексов: False
StratifiedKFold
Фолд 1 - размер train: наш=32900, sklearn=32901
Фолд 1 - размер test: наш=16452, sklearn=16451

Распределение классов в train (наша): Counter({np.int64(0): 32900})
Распределение классов в train (sklearn): Counter({np.int64(0): 32900, np.int64(2): 1})
Распределение классов в test (наша): Counter({np.int64(0): 16451, np.int64(2): 1})
Распределение классов в test (sklearn): Counter({np.int64(0): 16451})
GroupKFold

Фолд 1 - размер train: наш=31669, sklearn=33303
Фолд 1 - размер test: наш=17683, sklearn=16049
Уникальные группы в test (наша): {1, 4, 7}
Уникальные группы в test (sklearn): {1, 5, 6, 7, 8}
TimeSeriesSplit
Фолд: 1
Train размер: наш=12338, sklearn=12338
Test размер: наш=12338, sklearn=12338
Train dates: 2016-04-01 22:12:41 ... 2016-04-23 02:10:2

/opt/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Для k-fold разные стратегии распределения остатка

Лучший метод валидации: Stratified K-Fold

Обоснование выбора:
1. Целевая переменная имеет сильно несбалансированное распределение 

2. Обычный K-Fold мог бы создать фолды, где редкие классы отсутствуют в обучающей или тестовой выборке, что привело бы к некорректной оценке модели.

3. Stratified K-Fold сохраняет пропорции классов в каждом фолде:
   - Класс 0: ~99.99% в каждом фолде
   - Редкие классы: распределены пропорционально их исходной частоте
   
4. Это гарантирует, что модель обучается на всех типах объектов и валидируется на всех типах объектов, давая более надежную оценку.

5. Сравнение с sklearn подтверждает корректность реализации:
   - Распределения классов в train и test практически идентичны
   - Небольшие различия связаны с сильным дисбалансом данных (предупреждение sklearn о классе с 1 объектом)

## 6. Отбор признаков

In [33]:
X.head()

,bathrooms,bedrooms,created,has_Elevator,has_CatsAllowed,has_HardwoodFloors,has_DogsAllowed,has_Doorman,has_Dishwasher,has_NoFee,...,has_LaundryinUnit,has_RoofDeck,has_OutdoorSpace,has_DiningRoom,has_HighSpeedInternet,has_Balcony,has_SwimmingPool,has_LaundryInBuilding,has_NewConstruction,has_Terrace
4,1.0,1,2016-06-16 05:55:27,0,1,1,1,0,1,0,...,0,0,0,1,0,0,0,0,0,0
6,1.0,2,2016-06-01 05:44:33,1,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
9,1.0,2,2016-06-14 15:19:59,1,0,1,0,1,1,0,...,1,0,0,0,0,0,0,0,0,0
10,1.5,3,2016-06-24 07:54:24,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,1.0,0,2016-06-28 03:50:23,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_date_split(X, y, 'created',
                                                                           val_date = 0.2, test_date=0.2)
X_train = X_train.drop('created', axis=1)
X_val = X_val.drop('created', axis=1)
X_test = X_test.drop('created', axis=1)

numeric_features = ['bathrooms', 'bedrooms']
scaler = MinMaxScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

def fit_lasso(X_train, X_val, y_train, y_val):
    alphas = np.logspace(-2, 1, 300)
    best_score = -np.inf
    best_model = None

    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000, random_state=42)
        lasso.fit(X_train, y_train)

        val_score = lasso.score(X_val, y_val)

        if val_score > best_score:
            best_score = val_score
            best_model = lasso

    feature_names = X_train.columns.to_list()
    coef = best_model.coef_

    feature_imp = pd.DataFrame({
        'feature': feature_names, 'abs_coef': np.abs(coef)
    }).sort_values('abs_coef', ascending=False)

    return feature_imp, best_model

feature_imp, model = fit_lasso(X_train, X_val, y_train, y_val)
feature_imp.head(10)

,feature,abs_coef
0,bathrooms,24070.581979
1,bedrooms,2870.832158
6,has_Doorman,1139.622549
21,has_Terrace,587.775552
12,has_LaundryinUnit,567.522554
19,has_LaundryInBuilding,484.803217
9,has_LaundryinBuilding,398.695483
5,has_DogsAllowed,364.639126
16,has_HighSpeedInternet,334.146143
8,has_NoFee,272.941678


In [35]:
def calc_metrics(model, X_train, X_val, X_test, y_train, y_val, y_test, name, existing):
    preds = {
        "train": model.predict(X_train),
        "val": model.predict(X_val),
        "test": model.predict(X_test),
    }
    y = {"train": y_train, "val": y_val, "test": y_test}
    rows = []
    for split in ["train", "val", "test"]:
        mae = mean_absolute_error(y[split], preds[split])
        rmse = root_mean_squared_error(y[split], preds[split])
        r2 = r2_score(y[split], preds[split])
        rows.append({"calculation": name,"split": split, "MAE": mae, "RMSE": rmse, "R2": r2})
    new_df =  pd.DataFrame(rows)

    if existing is not None:
        result_df = pd.concat([existing, new_df], ignore_index=True)
    else:
        result_df = new_df

    return result_df

In [36]:
metrics_df = calc_metrics(model, X_train, X_val, X_test, y_train, y_val, y_test, "lasso", None)
metrics_df

,calculation,split,MAE,RMSE,R2
0,lasso,train,975.721250,8976.160639,0.033603
1,lasso,val,922.169961,1999.818636,0.442541
2,lasso,test,1463.282936,46621.843179,0.001585


In [37]:
top10_features = feature_imp['feature'].head(10).to_list()

In [38]:
X_train_top10 = X_train[top10_features]
X_val_top10 = X_val[top10_features]
X_test_top10 = X_test[top10_features]

feature_imp, model = fit_lasso(X_train_top10, X_val_top10, y_train, y_val)
metrics_df = calc_metrics(model, X_train_top10, X_val_top10, X_test_top10, 
                          y_train, y_val, y_test, "lasso_top10", metrics_df)
metrics_df

,calculation,split,MAE,RMSE,R2
0,lasso,train,975.721250,8976.160639,0.033603
1,lasso,val,922.169961,1999.818636,0.442541
2,lasso,test,1463.282936,46621.843179,0.001585
3,lasso_top10,train,968.334621,8977.634743,0.033286
4,lasso_top10,val,917.237445,1999.355356,0.442799
5,lasso_top10,test,1457.369471,46623.524631,0.001513


### Простой отбор признаков

In [39]:
def simple_feature_selection(X, y, ratio_threshold=0.5):
    if len(X) != len(y):
        raise ValueError(f"Длины X ({len(X)}) и y ({len(y)}) не совпадают")
    
    if not isinstance(ratio_threshold, (int, float)):
        raise TypeError("ratio_threshold must be a number")
    
    if not (0 < ratio_threshold < 1):
        raise ValueError("ratio must be between 0 and 1")
    
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    X_numeric = X[numeric_cols]

    features = []
    for col in X_numeric.columns:
        nan_ratio = X[col].isna().mean()
        if nan_ratio > ratio_threshold:
            continue
        
        corr = X[col].corr(y)

        features.append({"feature" : col, "nan_ratio": nan_ratio, 
                         "correlation" : corr, "abs_corr" : abs(corr)})
        
    features_df = pd.DataFrame(features)

    features_df = features_df.sort_values("abs_corr", ascending=False)

    return features_df

In [40]:
t0 = time.perf_counter()
features_select = simple_feature_selection(X, y)
top10_features = features_select["feature"].head(10).to_list()
top10_features

['bathrooms',
 'bedrooms',
 'has_Doorman',
 'has_Elevator',
 'has_LaundryinUnit',
 'has_DiningRoom',
 'has_FitnessCenter',
 'has_Terrace',
 'has_Dishwasher',
 'has_DogsAllowed']

In [41]:
X_train_top10 = X_train[top10_features]
X_val_top10 = X_val[top10_features]
X_test_top10 = X_test[top10_features]

feature_imp, model = fit_lasso(X_train_top10, X_val_top10, y_train, y_val)
metrics_df = calc_metrics(model, X_train_top10, X_val_top10, X_test_top10, 
                          y_train, y_val, y_test, "lasso_simple_top10", metrics_df)
t1 = time.perf_counter()
lasso_time = t1 - t0
metrics_df

,calculation,split,MAE,RMSE,R2
0,lasso,train,975.721250,8976.160639,0.033603
1,lasso,val,922.169961,1999.818636,0.442541
2,lasso,test,1463.282936,46621.843179,0.001585
3,lasso_top10,train,968.334621,8977.634743,0.033286
4,lasso_top10,val,917.237445,1999.355356,0.442799
5,lasso_top10,test,1457.369471,46623.524631,0.001513
6,lasso_simple_top10,train,974.334344,8981.092352,0.032541
7,lasso_simple_top10,val,927.525230,2010.138445,0.436773
8,lasso_simple_top10,test,1462.855847,46626.024389,0.001406


### Permutation importance

In [42]:
def permutation_importance(model, X_val, y_val, n_repeat=10):
    base_pred = model.predict(X_val)
    base_score = mean_absolute_percentage_error(y_val, base_pred)

    means = []
    stds = []

    for col in X_val.columns:
        scores = []
        for _ in range(n_repeat):
            X_permuted = X_val.copy()
            X_permuted[col] = np.random.permutation(X_permuted[col])

            pred = model.predict(X_permuted)
            score = mean_absolute_percentage_error(y_val, pred)

            importance = score - base_score
            scores.append(importance)

        means.append(np.mean(scores))
        stds.append(np.std(scores))

    
    results_df = pd.DataFrame({
        "feature" : X_val.columns, "mean" : means, "std" : stds
    })

    results_df = results_df.sort_values("mean", ascending=False)

    return results_df

In [43]:
lasso = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso.fit(X_train, y_train)

t0 = time.perf_counter()
perm_imp = permutation_importance(lasso, X_val, y_val)
top10_features = perm_imp.head(10)['feature'].tolist()
perm_imp.head(10)

,feature,mean,std
0,bathrooms,0.134846,0.002303
6,has_Doorman,0.074439,0.001344
1,bedrooms,0.056900,0.001673
12,has_LaundryinUnit,0.012325,0.000732
5,has_DogsAllowed,0.004913,0.000653
2,has_Elevator,0.004755,0.000385
3,has_CatsAllowed,0.003650,0.000187
21,has_Terrace,0.003537,0.000330
9,has_LaundryinBuilding,0.002803,0.000430
19,has_LaundryInBuilding,0.002096,0.000288


In [44]:
X_train_top10 = X_train[top10_features]
X_val_top10 = X_val[top10_features]
X_test_top10 = X_test[top10_features]

feature_imp, model = fit_lasso(X_train_top10, X_val_top10, y_train, y_val)
metrics_df = calc_metrics(model, X_train_top10, X_val_top10, X_test_top10, 
                          y_train, y_val, y_test, "lasso_perm_top10", metrics_df)
t1 = time.perf_counter()
perm_time = t1 - t0
metrics_df

,calculation,split,MAE,RMSE,R2
0,lasso,train,975.721250,8976.160639,0.033603
1,lasso,val,922.169961,1999.818636,0.442541
2,lasso,test,1463.282936,46621.843179,0.001585
3,lasso_top10,train,968.334621,8977.634743,0.033286
4,lasso_top10,val,917.237445,1999.355356,0.442799
5,lasso_top10,test,1457.369471,46623.524631,0.001513
6,lasso_simple_top10,train,974.334344,8981.092352,0.032541
7,lasso_simple_top10,val,927.525230,2010.138445,0.436773
8,lasso_simple_top10,test,1462.855847,46626.024389,0.001406
9,lasso_perm_top10,train,964.705441,8979.311636,0.032925


### SHAP

In [45]:
lasso = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso.fit(X_train, y_train)

def shap_feature_selection(model, X_train, X_val):
    explainer = shap.LinearExplainer(model, X_train)
    shap_values = explainer.shap_values(X_val)
    shap_importance = np.abs(shap_values).mean(axis=0)

    importance_df = pd.DataFrame({
        "feature": X_train.columns, "importance": shap_importance
    })

    importance_df = importance_df.sort_values(
        "importance", ascending=False
    )

    return importance_df

t0 = time.perf_counter()
shap_imp = shap_feature_selection(lasso, X_train, X_val)
top10_features = shap_imp["feature"].head(10).to_list()
shap_imp.head(10)

,feature,importance
0,bathrooms,777.696848
6,has_Doorman,560.320193
1,bedrooms,381.155428
12,has_LaundryinUnit,181.880916
5,has_DogsAllowed,181.069372
9,has_LaundryinBuilding,171.863202
8,has_NoFee,124.831411
4,has_HardwoodFloors,105.016355
2,has_Elevator,96.439664
3,has_CatsAllowed,78.591070


In [46]:
X_train_top10 = X_train[top10_features]
X_val_top10 = X_val[top10_features]
X_test_top10 = X_test[top10_features]


feature_imp, model = fit_lasso(X_train_top10, X_val_top10, y_train, y_val)
metrics_df = calc_metrics(model, X_train_top10, X_val_top10, X_test_top10, 
                          y_train, y_val, y_test, "lasso_shap_top10", metrics_df)
t1 = time.perf_counter()
shap_time = t1 - t0
display(metrics_df)
print(f"simple split time: {lasso_time}")
print(f"permutation importance time: {perm_time}")
print(f"shap time: {shap_time}")

,calculation,split,MAE,RMSE,R2
0,lasso,train,975.721250,8976.160639,0.033603
1,lasso,val,922.169961,1999.818636,0.442541
2,lasso,test,1463.282936,46621.843179,0.001585
3,lasso_top10,train,968.334621,8977.634743,0.033286
4,lasso_top10,val,917.237445,1999.355356,0.442799
5,lasso_top10,test,1457.369471,46623.524631,0.001513
6,lasso_simple_top10,train,974.334344,8981.092352,0.032541
7,lasso_simple_top10,val,927.525230,2010.138445,0.436773
8,lasso_simple_top10,test,1462.855847,46626.024389,0.001406
9,lasso_perm_top10,train,964.705441,8979.311636,0.032925


simple split time: 3.779970458999742
permutation importance time: 12.513668166007847
shap time: 12.191200166998897


- скорость: Lasso < permutation ≈ SHAP;

- стабильность: порядок важных фич у методов похож, особенно у permutation и SHAP.

## 7. Оптимизация гиперпараметров

### Grid Search

In [47]:
class MyGridSearch:
    def __init__(self, model, param_grid, cv=5):
        self.model = model
        self.param_grid = param_grid
        self.cv = cv
        self.best_params = None
        self.best_score = -np.inf
        self.best_model = None

    def generate_param_comb(self):
        keys = self.param_grid.keys()
        values = self.param_grid.values()

        combinations = []

        for comb in product(*values):
            params = dict(zip(keys, comb))
            combinations.append(params)

        return combinations


    def fit(self, X, y):
        params_comb = self.generate_param_comb()

        for params in params_comb:

            scores = self.cross_validation(X, y, params)
            mean_score = np.mean(scores)

            if mean_score > self.best_score:

                self.best_score = mean_score
                self.best_params = params

        self.best_model = clone(self.model)
        self.best_model.set_params(**self.best_params)
        self.best_model.fit(X, y)

        return self


    def cross_validation(self, X, y, params):
        folds = kfold_split(X, k=self.cv)
        scores = []

        for train_idx, test_idx in folds:

            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

            model = clone(self.model)
            model.set_params(**params)

            model.fit(X_train, y_train)

            score = model.score(X_test, y_test)

            scores.append(score)

        return scores

In [48]:
param_grid = {
    "alpha": [0.001, 0.01, 0.1],
    "l1_ratio": [0.2, 0.5, 0.8]
}

grid = MyGridSearch(
    ElasticNet(max_iter=10000),
    param_grid,
    cv=5
)

grid.fit(X_train, y_train)

print(grid.best_params)
print(grid.best_score)

{'alpha': 0.001, 'l1_ratio': 0.2}
0.3073044094377365


### Random Grid Search

In [49]:
class MyRandomSearch:
    def __init__(self, model, param_grid, cv=5, n_iter=20, random_state=42):
        self.model = model
        self.param_grid = param_grid
        self.cv = cv
        self.n_iter = n_iter
        self.best_params = None
        self.best_score = -np.inf
        self.best_model = None

        if random_state:
            np.random.seed(random_state)

    def select_params(self):
        params = {}

        for key, values in self.param_grid.items():
            params[key] = np.random.choice(values)

        return params


    def fit(self, X, y):
        for _ in range(self.n_iter):
            params = self.select_params()

            scores = self.cross_validation(X, y, params)
            mean_score = np.mean(scores)

            if mean_score > self.best_score:

                self.best_score = mean_score
                self.best_params = params

        self.best_model = clone(self.model)
        self.best_model.set_params(**self.best_params)
        self.best_model.fit(X, y)

        return self


    def cross_validation(self, X, y, params):

        folds = kfold_split(X, k=self.cv)
        scores = []

        for train_idx, test_idx in folds:

            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

            model = clone(self.model)
            model.set_params(**params)

            model.fit(X_train, y_train)

            score = model.score(X_test, y_test)

            scores.append(score)

        return scores

In [50]:
param_dist = {
    "alpha": np.logspace(-4, 1, 50),
    "l1_ratio": np.linspace(0, 1, 50)
}

random_search = MyRandomSearch(
    ElasticNet(max_iter=10000),
    param_dist,
    n_iter=30,
    cv=5,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params)
print("Best score:", random_search.best_score)

Best params: {'alpha': np.float64(0.0008286427728546842), 'l1_ratio': np.float64(0.8979591836734693)}
Best score: 0.3582866056448715


### Optuna

In [51]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0, 1)

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10000,
        random_state=42
    )

    folds = kfold_split(X_train, k=5)

    scores = []
    for train_indx, val_indx in folds:
        X_tr, y_tr = X_train.iloc[train_indx], y_train.iloc[train_indx]
        X_val, y_val = X_train.iloc[val_indx], y_train.iloc[val_indx]

        model.fit(X_tr, y_tr)

        score = model.score(X_val, y_val)
        scores.append(score)

    return np.mean(scores)

In [52]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-06-14 22:01:07,649] A new study created in memory with name: no-name-1ecb5daa-304e-4ad0-be8a-fb5950a340c0
[I 2026-06-14 22:01:07,713] Trial 0 finished with value: 0.019641550626684844 and parameters: {'alpha': 9.830046132225783, 'l1_ratio': 0.6800872626877341}. Best is trial 0 with value: 0.019641550626684844.
[I 2026-06-14 22:01:08,785] Trial 1 finished with value: 0.28479775837043037 and parameters: {'alpha': 0.00023095841904495238, 'l1_ratio': 0.48957444620932344}. Best is trial 1 with value: 0.28479775837043037.
[I 2026-06-14 22:01:08,851] Trial 2 finished with value: 0.015628842969604363 and parameters: {'alpha': 4.416802890627236, 'l1_ratio': 0.35010504468901416}. Best is trial 1 with value: 0.28479775837043037.
[I 2026-06-14 22:01:09,814] Trial 3 finished with value: 0.2172850520557903 and parameters: {'alpha': 0.0011413261595675007, 'l1_ratio': 0.30391412146766894}. Best is trial 1 with value: 0.28479775837043037.
[I 2026-06-14 22:01:10,871] Trial 4 finished with value:

Best params: {'alpha': 0.0004953275682774604, 'l1_ratio': 0.5237693671229239}
Best score: 0.3285106004831544


Для оптимизации гиперпараметров ElasticNet использовалась схема перекрестной проверки с помощью Optuna. По сравнению с поиском по сетке и случайным поиском, Optuna более эффективно исследует пространство параметров, используя байесовскую оптимизацию. В результате она часто позволяет добиться такой же или даже лучшей производительности за меньшее количество итераций. 

